In [ ]:
import pandas as pd
import numpy as np
import sys
from scipy.stats import skew, kurtosis
from sklearn.linear_model import LinearRegression
sys.path.append(r"D:\HopeAI\Assignments\MyModules")

from data_analysis_utils import Preprocessing

stocks_df = pd.read_csv(r"D:\HopeAI\Assignments\ML and DS Capstone\1. Data Collection\nifty300_stocks.csv")
nifty_df = pd.read_csv(r'D:\HopeAI\Assignments\ML and DS Capstone\1. Data Collection\NIFTY50.csv')
stocks_df


,Date,Close,High,Low,Open,Volume,Stock,Sector,Industry,Market Cap,Dividend Yield,Dividend Rate,PE Ratios,ROE Values,Cap Category
0,2022-01-03,2165.453613,2227.458517,2145.963741,2210.619393,113948,ABB,Industrials,Specialty Industrial Machinery,1090798354432,0.008344,43.27,60.523220,0.249980,Large Cap
1,2022-01-04,2179.494141,2197.069423,2150.283666,2183.568783,353471,ABB,Industrials,Specialty Industrial Machinery,1090798354432,0.008344,43.27,60.523220,0.249980,Large Cap
2,2022-01-05,2185.826904,2209.097047,2160.102100,2189.116247,222632,ABB,Industrials,Specialty Industrial Machinery,1090798354432,0.008344,43.27,60.523220,0.249980,Large Cap
3,2022-01-06,2184.894287,2246.260827,2160.151346,2174.633850,423469,ABB,Industrials,Specialty Industrial Machinery,1090798354432,0.008344,43.27,60.523220,0.249980,Large Cap
4,2022-01-07,2186.710693,2222.892259,2176.302833,2196.381916,111305,ABB,Industrials,Specialty Industrial Machinery,1090798354432,0.008344,43.27,60.523220,0.249980,Large Cap
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
200426,2024-12-23,2573.374268,2596.341908,2403.263845,2551.405221,1014490,ZENTEC,Industrials,Aerospace & Defense,125330620416,0.001409,2.00,49.351982,0.147664,Small Cap
200427,2024-12-24,2561.341064,2623.303809,2519.100532,2593.346019,632541,ZENTEC,Industrials,Aerospace & Defense,125330620416,0.001409,2.00,49.351982,0.147664,Small Cap
200428,2024-12-26,2461.032471,2558.395289,2436.616920,2558.395289,643354,ZENTEC,Industrials,Aerospace & Defense,125330620416,0.001409,2.00,49.351982,0.147664,Small Cap
200429,2024-12-27,2400.867188,2474.613377,2344.796289,2470.918388,428550,ZENTEC,Industrials,Aerospace & Defense,125330620416,0.001409,2.00,49.351982,0.147664,Small Cap


In [5]:
stocks_df['Date'] = pd.to_datetime(stocks_df['Date'])

# Filter to last 3 years (assuming today is the current date)
latest_year = stocks_df['Date'].dt.year.max()
three_years = [latest_year-2, latest_year-1, latest_year]
df_3y = stocks_df[stocks_df['Date'].dt.year.isin(three_years)]
# Find stocks with data in all 3 years
stocks_full_3y = (
    df_3y.groupby('Stock')['Date'].apply(lambda x: x.dt.year.nunique())
    .loc[lambda count: count == 3]
    .index
)

# Filter DataFrame to only those stocks
df_3y_full = df_3y[df_3y['Stock'].isin(stocks_full_3y)]
nifty_returns = nifty_df[['Date', 'Close']].copy()
nifty_df['Date'] = pd.to_datetime(nifty_df['Date'])
nifty_returns = nifty_df[['Date','Close']].set_index('Date')['Close'].pct_change().dropna()

def calc_metrics(group):
    sector = group['Sector'].unique()[0]  # use [0] instead of .item() for robustness
    industry = group['Industry'].unique()[0]
    market_cap = group['Market Cap'].unique()[0]
    cap_category = group['Cap Category'].unique()[0]
    
    group = group.sort_values('Date')
    group['daily_return'] = group['Close'].pct_change()
    avg_daily_return = group['daily_return'].mean()
    std_daily = group['daily_return'].std()
    trading_days = group['Date'].dt.date.nunique()  # number of trading days in this period
    
    annualized_return = (1 + avg_daily_return) ** trading_days - 1
    annualized_volatility = std_daily * np.sqrt(trading_days)
    risk_free_rate = 0.05  # (0.05 or 5% annual for INR/India government bonds)
    sharpe_ratio = (annualized_return - risk_free_rate) / annualized_volatility if annualized_volatility != 0 else np.nan
    max_drawdown = ((group['Close'].cummax() - group['Close']) / group['Close'].cummax()).max()
    avg_volume = group['Volume'].mean()
    volume_spike_pct = (group['Volume'] > 2 * avg_volume).mean()
    
    # Market relationship: Use group directly instead of 'year_data'
    group = group.set_index('Date').sort_index()
    daily_returns = group['Close'].pct_change().dropna()
    # nifty_returns already covers the whole period, match dates
    nifty_year = nifty_returns.loc[daily_returns.index.min(): daily_returns.index.max()]
    combined = pd.DataFrame({
        'Stock_Return': daily_returns,
        'NIFTY_Return': nifty_year
    }).dropna()
    
    if len(combined) > 30:  # require at least 30 days
        cov_matrix = np.cov(combined['Stock_Return'], combined['NIFTY_Return'])
        beta = cov_matrix[0,1] / cov_matrix[1,1] if cov_matrix[1,1] != 0 else np.nan
        corr_nifty = combined['Stock_Return'].corr(combined['NIFTY_Return'])
    else:
        beta = np.nan
        corr_nifty = np.nan

    return pd.Series({
        'Sector': sector,   
        'Industry': industry,
        'MarketCap': market_cap,
        'Cap Category': cap_category,
        'Annualized_Return': annualized_return,
        'Annualized_Volatility': annualized_volatility,
        'Sharpe_Ratio': sharpe_ratio,
        'Max_Drawdown': max_drawdown*100, # Percentage
        'Avg_Volume': avg_volume,
        'Volume_Spike_Pct': volume_spike_pct*100,  # Percentage
        'Beta_vs_NIFTY': beta,
        'Corr_with_NIFTY': corr_nifty
    })

features_df = df_3y_full.groupby('Stock').apply(calc_metrics, include_groups=False).reset_index()
features_df


,Stock,Sector,Industry,MarketCap,Cap Category,Annualized_Return,Annualized_Volatility,Sharpe_Ratio,Max_Drawdown,Avg_Volume,Volume_Spike_Pct,Beta_vs_NIFTY,Corr_with_NIFTY
0,360ONE,Financial Services,Asset Management,445803167744,Mid Cap,3.524515,0.589975,5.889254,30.097756,4.899030e+05,11.111111,0.639538,0.260950
1,AARTIIND,Basic Materials,Specialty Chemicals,138456645632,Small Cap,-0.517008,0.585279,-0.968783,63.172265,1.300992e+06,7.317073,1.252114,0.514999
2,ABB,Industrials,Specialty Industrial Machinery,1090798354432,Large Cap,2.615783,0.571198,4.491934,26.442369,3.516467e+05,6.368564,0.862478,0.363485
3,ABCAPITAL,Financial Services,Financial Conglomerates,781306822656,Mid Cap,0.639011,0.573486,1.027072,35.630066,5.000453e+06,9.214092,1.392779,0.584635
4,ABREL,Basic Materials,Paper & Paper Products,178644762624,Small Cap,2.476973,0.697101,3.481524,39.647621,3.216849e+05,10.704607,1.227745,0.423973
...,...,...,...,...,...,...,...,...,...,...,...,...,...
262,WIPRO,Technology,Information Technology Services,2567080247296,Large Cap,-0.058463,0.432499,-0.250782,50.015772,1.370024e+07,5.555556,1.081205,0.601794
263,WOCKPHARMA,Healthcare,Drug Manufacturers - Specialty & Generic,225707163648,Small Cap,4.216138,0.828349,5.029448,63.056593,1.059813e+06,10.027100,0.982579,0.285548
264,YESBANK,Financial Services,Banks - Regional,754220466176,Mid Cap,0.777135,0.707369,1.027943,39.267516,1.655482e+08,10.569106,1.091840,0.371568
265,ZENTEC,Industrials,Aerospace & Defense,125330620416,Small Cap,14.446618,0.809376,17.787296,34.040224,5.798867e+05,10.840108,0.969627,0.288389


In [6]:
features_df.columns


Index(['Stock', 'Sector', 'Industry', 'MarketCap', 'Cap Category',
       'Annualized_Return', 'Annualized_Volatility', 'Sharpe_Ratio',
       'Max_Drawdown', 'Avg_Volume', 'Volume_Spike_Pct', 'Beta_vs_NIFTY',
       'Corr_with_NIFTY'],
      dtype='object')

In [7]:
features_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 267 entries, 0 to 266
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Stock                  267 non-null    object 
 1   Sector                 267 non-null    object 
 2   Industry               267 non-null    object 
 3   MarketCap              267 non-null    int64  
 4   Cap Category           267 non-null    object 
 5   Annualized_Return      267 non-null    float64
 6   Annualized_Volatility  267 non-null    float64
 7   Sharpe_Ratio           267 non-null    float64
 8   Max_Drawdown           267 non-null    float64
 9   Avg_Volume             267 non-null    float64
 10  Volume_Spike_Pct       267 non-null    float64
 11  Beta_vs_NIFTY          267 non-null    float64
 12  Corr_with_NIFTY        267 non-null    float64
dtypes: float64(8), int64(1), object(4)
memory usage: 27.2+ KB


In [8]:
quan, qual = Preprocessing.quanQual(features_df)

print(quan, qual)


Index(['MarketCap', 'Annualized_Return', 'Annualized_Volatility',
       'Sharpe_Ratio', 'Max_Drawdown', 'Avg_Volume', 'Volume_Spike_Pct',
       'Beta_vs_NIFTY', 'Corr_with_NIFTY'],
      dtype='object') Index(['Stock', 'Sector', 'Industry', 'Cap Category'], dtype='object')


In [ ]:
import pickle
features_df.to_csv('PreStockFeatures.csv', index=False)
filename='pre_dataset.pkl'
pickle.dump(features_df, open(filename,'wb'))
